# 分支发散与占用率调优

协同算子的性能上限，取决于"最弱的那段"。本节给出两个指标与对应调优动作。


## 1. 指标一：分支发散率

第 6 章的结论定量展开：设 SIMT 段内某分支处，Warp 内走 A 路的比例为 p：

```text
若编译结果保留两个分支，简化模型中该分支时间 ≈ T_A × [p > 0] + T_B × [p < 1]
发散代价 = 相对无发散（全走一路）的额外时间
```

调优动作（按优先级）：

1. **数据排布对齐**：让同 Warp 的线程面对同质数据（如按阈值分区排布输入，Warp 内 p 接近 1 或 0）；
2. **无分支改写**：条件标量乘/比较/最小最大（第 6 章对策二）——选择段常见模式（有限输入下的 max/min；若用比较乘法，须先核对目标版本是否支持所需的布尔类型转换）优先无分支化；
3. **接受发散**：p 长期居中（约 0.5）且分支体极短时，发散代价 < 改写成本，保留分支可能更合适——**用第 8 章的 profiler 数据说话**，不要凭感觉。


## 2. 指标二：线程工作分配与硬件 occupancy

应区分三个概念：候选命中率、执行有用工作的 lane 比例、硬件 occupancy（驻留 Warp/线程相对硬件上限）。候选未命中也执行了加载与比较，不能把命中率直接当 occupancy。

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">现象</th><th style="text-align:left;">可尝试的调整</th><th style="text-align:left;">限制</th></tr>
<tr><td>数据量小于启动线程数</td><td>减少 Block 或线程数</td><td>grid-stride 保证覆盖，不自动消除空闲线程</td></tr>
<tr><td>命中稀疏</td><td>每线程扫描多个候选，或重排后续工作</td><td>加载/比较仍需执行，线程工作量和寄存器压力可能上升</td></tr>
<tr><td>Block 过大</td><td>扫描线程配置并查看寄存器/驻留资源</td><td>32 对齐不保证最高吞吐</td></tr>
</table>

独立命中率 p=10% 时，每线程查 10 个候选，期望命中 1 个，但至少命中一次的概率仅为 1−(1−p)¹⁰≈65.1%，并非所有线程都命中；这也不是硬件 occupancy。


## 3. 协同段间的配平

先确认两段使用的资源：同一 AIV 上的 SIMD Vector 与 SIMT 都占 V 流水，不能仅凭模式不同就套用 max(T_simd,T_simt)。只有不同资源可重叠且依赖/缓冲允许时，才使用第 8 章的理想两阶段模型。选择段快、计算段慢时，选择段可"预取流水"（第 k+1 轮的选择与第 k 轮的计算重叠，第 5 章错位法套用到段间）；反向则考虑选择段并行化（多 Block 分担）。


## 4. 调优决策树

```text
协同算子慢 → profiler 看两段占用
├── SIMT 段慢 → 看发散率
│     ├── 发散高（Warp 内分裂大）→ 排布对齐 / 无分支改写
│     └── 占用低（线程空转多）→ grid-stride / 多候选摊薄
├── SIMD 段慢 → 回第 8 章四问（切分/缓冲/流水/搬运）
└── 段间空等 → 交接开销大？→ 载体换 UB Tile
                稳态失衡？→ 段间预取流水
```


### 小候选排名与大规模选择的成本不同

32 选 8 的严格排名需要 32×32 次比较，每个候选得到唯一 rank，写出顺序稳定，容易作为教学正确性基线。若直接扩到 4096 个候选，比较数量变为 16777216，不能沿用“小规模下可接受”的判断。

测量至少拆出选择、紧凑化、稠密处理和交接开销。筛选后有效行数减少，原子计数和全候选扫描仍然存在；同一 AIV 上两种编程模式也共用 V 资源。只有不同资源和依赖允许重叠时，才用 max 模型估计稳态下界。


## 5. 课后练习

1. （计算）每个候选独立以 5% 概率命中；每线程查 20 个候选，期望命中数与至少一次命中的概率各是多少？为什么不能由此直接推导硬件 occupancy？
2. （实践）为第 7 章实践三（Scatter 仲裁）估算发散代价：仲裁分支处 Warp 内命中比例不定——你会先做哪个调优动作？为什么？

In [ ]:
!cat ./answer/09.04_answer.txt

## 6. 本节小结

- 两指标：**发散情况**（按实际产码判断分支路径）与**线程工作分配**，并区分硬件 occupancy；
- 发散三动作：排布对齐 > 无分支改写 > 必要时接受（数据说话）；
- 候选命中率不等于线程 occupancy；多候选循环的成本与收益须实测；
- 先查资源与依赖，再判断能否流水重叠；同一 V 流水不能假定两段并行。